In [1]:
pip list


Package                   Version
------------------------- -----------
absl-py                   2.3.1
albucore                  0.0.24
albumentations            2.0.8
altair                    5.5.0
annotated-types           0.7.0
anyio                     4.11.0
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.0
astunparse                1.6.3
async-lru                 2.0.5
attrs                     25.4.0
babel                     2.17.0
beautifulsoup4            4.14.2
bleach                    6.2.0
blinker                   1.9.0
branca                    0.8.2
cachetools                6.2.1
certifi                   2025.10.5
cffi                      2.0.0
cftime                    1.6.5
charset-normalizer        3.4.4
click                     8.1.8
cloudpickle               3.1.1
colorama                  0.4.6
comm                      0.2.3
contourpy                 1.3.2
cycler               

In [2]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA

In [8]:

plant_path = "../Data/Raw/PlantVillage"
IMG_SIZE = (128, 128)
data, labels = [], []

print("Starting data loading...")

for folder in os.listdir(plant_path):
    folder_path = os.path.join(plant_path, folder)
    print(f"Processing folder: {folder}")
    
    for file in os.listdir(folder_path):
        img_path = os.path.join(folder_path, file)
        img = cv2.imread(img_path)
        
        # --- THIS IS THE FIX ---
        # Check if the image was loaded successfully
        if img is not None:
            try:
                # 1. Convert color
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                # 2. Resize
                img = cv2.resize(img, IMG_SIZE)
                # 3. Append data and label
                data.append(img)
                labels.append(folder)
            except Exception as e:
                print(f"Error processing file {img_path}: {e}")
        else:
            print(f"Warning: Skipping file, could not read: {img_path}")

print("Data loading complete.")

# --- Rest of your code (this part was correct) ---
print("Converting to NumPy arrays and normalizing...")
data = np.array(data) / 255.0
labels = np.array(labels)

print("Encoding labels...")
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

print("Splitting data into train and test sets...")
X_train_rgb, X_test_rgb, y_train_rgb, y_test_rgb = train_test_split(
    data, labels_encoded, test_size=0.2, random_state=42, stratify=labels_encoded
)

print("Done.")
print(f"X_train shape: {X_train_rgb.shape}, y_train shape: {y_train_rgb.shape}")
print(f"X_test shape: {X_test_rgb.shape}, y_test shape: {y_test_rgb.shape}")

Starting data loading...
Processing folder: .ipynb_checkpoints
Processing folder: Pepper__bell___Bacterial_spot
Processing folder: Pepper__bell___healthy
Processing folder: Potato___Early_blight
Processing folder: Potato___healthy
Processing folder: Potato___Late_blight
Processing folder: Tomato_Bacterial_spot
Processing folder: Tomato_Early_blight
Processing folder: Tomato_healthy
Processing folder: Tomato_Late_blight
Processing folder: Tomato_Leaf_Mold
Processing folder: Tomato_Septoria_leaf_spot
Processing folder: Tomato_Spider_mites_Two_spotted_spider_mite
Processing folder: Tomato__Target_Spot
Processing folder: Tomato__Tomato_mosaic_virus
Processing folder: Tomato__Tomato_YellowLeaf__Curl_Virus
Data loading complete.
Converting to NumPy arrays and normalizing...
Encoding labels...
Splitting data into train and test sets...
Done.
X_train shape: (16318, 128, 128, 3), y_train shape: (16318,)
X_test shape: (4080, 128, 128, 3), y_test shape: (4080,)


In [22]:
import pandas as pd
import os

# --- Path to your CSV file ---
train_csv_path = "../Data/Raw/HyperLeaf2024/HyperLeaf2024/train.csv"

# Check if the file exists
if not os.path.exists(train_csv_path):
    print(f"ERROR: Cannot find train.csv at: {train_csv_path}")
else:
    # Load just the CSV
    train_labels_df = pd.read_csv(train_csv_path)
    
    print("--- CSV Data Report ---")
    
    # 1. Print the 'ImageId' column's data type
    print(f"Data type of 'ImageId' column: {train_labels_df['ImageId'].dtype}")
    
    # 2. Print the first 5 values from that column
    print("\nFirst 5 'ImageId' values:")
    print(train_labels_df['ImageId'].head())

--- CSV Data Report ---
Data type of 'ImageId' column: int64

First 5 'ImageId' values:
0    0
1    1
2    3
3    7
4    9
Name: ImageId, dtype: int64


In [29]:

train_csv_path = "../Data/Raw/HyperLeaf2024/HyperLeaf2024/train.csv"

# Check if the file exists
if not os.path.exists(train_csv_path):
    print(f"ERROR: Cannot find train.csv at: {train_csv_path}")
else:
    # Load just the CSV
    train_labels_df = pd.read_csv(train_csv_path)
    
    print("--- CSV Column Report ---")
    
    # 1. Print all column names
    print("The CSV file contains the following columns:")
    print(list(train_labels_df.columns))
    
    # 2. Print the first few rows to see the data
    print("\nHere's a preview of the data:")
    print(train_labels_df.head())

--- CSV Column Report ---
The CSV file contains the following columns:
['ImageId', 'GrainWeight', 'Gsw', 'PhiPS2', 'Fertilizer', 'Heerup', 'Kvium', 'Rembrandt', 'Sheriff']

Here's a preview of the data:
   ImageId  GrainWeight       Gsw    PhiPS2  Fertilizer  Heerup  Kvium  \
0        0         4004  0.067527  0.627443         0.0       0      1   
1        1         6753  0.062773  0.664247         0.5       0      0   
2        3         8082  0.123482  0.425904         1.0       0      0   
3        7         8520  0.007312  0.691565         1.0       0      0   
4        9         8520  0.129374  0.502281         1.0       0      0   

   Rembrandt  Sheriff  
0          0        0  
1          0        1  
2          1        0  
3          0        1  
4          0        1  


In [30]:

import tifffile as tiff  # <-- IMPORT THE NEW LIBRARY
from sklearn.decomposition import PCA

print("Processing HyperLeaf2024 (Nutrient) Dataset...")

# Path to your data
data_path = "../Data/Raw/HyperLeaf2024/HyperLeaf2024/"
img_folder = os.path.join(data_path, "images")
train_csv_path = os.path.join(data_path, "train.csv")

if not os.path.exists(train_csv_path):
    print(f"ERROR: Cannot find train.csv at: {train_csv_path}")
else:
    train_labels_df = pd.read_csv(train_csv_path)
    data, labels = [], []
    total_images = len(train_labels_df)
    
    print(f"Starting to load and process {total_images} TIFF images... (Using tifffile)")
    print("This will take a long time (10-30+ minutes)...")

    # Loop through the CSV file
    for index, row in train_labels_df.iterrows():
        
        # Print an update every 100 images
        if (index + 1) % 100 == 0:
            print(f"   ...processed {index + 1} / {total_images} images")
            
        img_id_int = int(row['ImageId'])
        img_id_str = str(img_id_int).zfill(5)
        img_path = os.path.join(img_folder, f"{img_id_str}.tiff")
        
        try:
            hyperspectral_cube = tiff.imread(img_path)
            
            if hyperspectral_cube is not None:
                
                if hyperspectral_cube.ndim != 3:
                    print(f"Warning: Skipping {img_path}, not a 3D cube. Shape was {hyperspectral_cube.shape}")
                    continue
                
                h, w, bands = hyperspectral_cube.shape
                
                # --- Preprocessing with PCA ---
                pixels_flat = hyperspectral_cube.reshape(-1, bands)
                pca = PCA(n_components=3)
                reduced_flat = pca.fit_transform(pixels_flat)
                reduced_image = reduced_flat.reshape(h, w, 3)
                resized_img = cv2.resize(reduced_image, (128, 128))

                data.append(resized_img)
                
                # --- THIS IS THE FIX ---
                # Changed 'fertilizer_content' to 'Fertilizer'
                labels.append(row['Fertilizer'])
                # ---
                
            else:
                print(f"Warning: Skipping file, could not read: {img_path}")
                
        except Exception as e:
            print(f"ERROR processing {img_path}: {e}")

    print(f"Finished processing {total_images} images.")
    print("HyperLeaf data loading complete.")

    # --- Process and Save your data ---
    X_hyper = np.array(data)
    y_hyper = np.array(labels)

    if X_hyper.size > 0:
        # Normalize the PCA output
        X_hyper = (X_hyper - np.min(X_hyper)) / (np.max(X_hyper) - np.min(X_hyper))
        print(f"Hyperspectral data shape: {X_hyper.shape}")
        print(f"Hyperspectral labels shape: {y_hyper.shape}")

        # Split this data
        X_train_hyper, X_test_hyper, y_train_hyper, y_test_hyper = train_test_split(
            X_hyper, y_hyper, test_size=0.2, random_state=42
        )

        # --- Save the processed hyperspectral data ---
        print("Saving processed hyperspectral data...")
        np.save("../data/processed/X_train_hyper.npy", X_train_hyper)
        np.save("../data/processed/y_train_hyper.npy", y_train_hyper)
        np.save("../data/processed/X_test_hyper.npy", X_test_hyper)
        np.save("../data/processed/y_test_hyper.npy", y_test_hyper)
        print("✅ HyperLeaf processing, splitting, and saving complete.")
    else:
        print("ERROR: No data was loaded. Check warnings.")

  

Processing HyperLeaf2024 (Nutrient) Dataset...
Starting to load and process 1590 TIFF images... (Using tifffile)
This will take a long time (10-30+ minutes)...
   ...processed 100 / 1590 images
   ...processed 200 / 1590 images
   ...processed 300 / 1590 images
   ...processed 400 / 1590 images
   ...processed 500 / 1590 images
   ...processed 600 / 1590 images
   ...processed 700 / 1590 images
   ...processed 800 / 1590 images
   ...processed 900 / 1590 images
   ...processed 1000 / 1590 images
   ...processed 1100 / 1590 images
   ...processed 1200 / 1590 images
   ...processed 1300 / 1590 images
   ...processed 1400 / 1590 images
   ...processed 1500 / 1590 images
Finished processing 1590 images.
HyperLeaf data loading complete.
Hyperspectral data shape: (1590, 128, 128, 3)
Hyperspectral labels shape: (1590,)
Saving processed hyperspectral data...
✅ HyperLeaf processing, splitting, and saving complete.


In [18]:
print("\nProcessing USGS Crop Spectral Library...")
usgs_path = "../Data/Raw/USGS_Crop_Library/"

try:
    # Find the .csv file
    csv_files = [f for f in os.listdir(usgs_path) if f.endswith('.csv')]
    
    if not csv_files:
        print(f"ERROR: No .csv file found in {usgs_path}")
    else:
        file_path = os.path.join(usgs_path, csv_files[0])
        print(f"Loading CSV: {file_path}")
        
        df = pd.read_csv(file_path)
        print("DataFrame Head:")
        print(df.head())

        # Find all columns that are reflectance bands (e.g., '...nm')
        bands = [col for col in df.columns if 'nm' in col.lower()]
        
        if not bands:
            print("Warning: No 'nm' columns found for spectral analysis.")
        else:
            print(f"Found {len(bands)} spectral bands.")
            
            # Plot the first principal component of the spectral signatures
            pca = PCA(n_components=3)
            reduced = pca.fit_transform(df[bands])
            
            plt.figure(figsize=(10, 4))
            plt.plot(reduced[:, 0])
            plt.title("Spectral Signature (PCA Component 1)")
            plt.xlabel("Sample Index")
            plt.ylabel("PCA Component 1 Value")
            plt.show()
        
        print("✅ USGS Crop Library processing complete.")

except FileNotFoundError:
    print(f"ERROR: Directory not found: {usgs_path}")
except Exception as e:
    print(f"An error occurred: {e}")


Processing USGS Crop Spectral Library...
Loading CSV: ../Data/Raw/USGS_Crop_Library/GHISACONUS_2008_001_speclib.csv
DataFrame Head:
   UniqueID Country  AEZ                          Image  Month  Year   jd  \
0      1466     USA    7  EO1H0440332012234110KD_SGS_01      8  2012  234   
1      1467     USA    7  EO1H0440332012234110KD_SGS_01      8  2012  234   
2      1469     USA    7  EO1H0440332012234110KD_SGS_01      8  2012  234   
3      1470     USA    7  EO1H0440332012234110KD_SGS_01      8  2012  234   
4      1476     USA    7  EO1H0440332012234110KD_SGS_01      8  2012  234   

         long        lat  Crop  ...     X2254     X2264     X2274     X2285  \
0 -121.663419  38.534516  corn  ...  8.432855  8.438398  8.585301  8.164735   
1 -121.671589  38.504744  corn  ...  8.682515  8.942595  9.179517  8.744633   
2 -121.597588  38.614056  corn  ...  6.841276  6.276240  6.150597  6.114164   
3 -121.687293  38.571702  corn  ...  6.679967  6.534779  6.430438  6.080465   
4 -121.62

In [31]:
print("\nSaving final processed PlantVillage (RGB) datasets...")

try:
    # Check if the variables from Step 2 exist before saving
    if 'X_train_rgb' in locals():
        np.save("../data/processed/X_train_rgb.npy", X_train_rgb)
        np.save("../data/processed/y_train_rgb.npy", y_train_rgb)
        np.save("../data/processed/X_test_rgb.npy", X_test_rgb)
        np.save("../data/processed/y_test_rgb.npy", y_test_rgb)
        
        print("All PlantVillage (RGB) arrays saved successfully to ../data/processed/")
    else:
        print("ERROR: 'X_train_rgb' not defined. Did Step 2 (PlantVillage) fail to run or was it skipped?")

except Exception as e:
    print(f"An error occurred during saving: {e}")

print("\n--- Notebook 01 Data Exploration Complete! ---")


Saving final processed PlantVillage (RGB) datasets...
All PlantVillage (RGB) arrays saved successfully to ../data/processed/

--- Notebook 01 Data Exploration Complete! ---
